In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
df = pd.read_csv("anime.csv")

print(df.head())
print(df.info())

   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.16   

   members  
0   200630  
1   793665  
2   114262  
3   673572  
4   151266  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  

In [3]:
df = df.dropna(subset=['genre', 'rating'])

In [4]:
# Convert numeric columns
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['members'] = pd.to_numeric(df['members'], errors='coerce')

# Fill ALL numeric NaN
df.fillna(df.median(numeric_only=True), inplace=True)

# Fill categorical NaN
df.fillna("Unknown", inplace=True)

In [5]:
cv = CountVectorizer(tokenizer=lambda x: x.split(','))
genre_matrix = cv.fit_transform(df['genre'])

E:\Users\SUGANYA P\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [6]:
scaler = MinMaxScaler()

numeric_features = scaler.fit_transform(
    df[['rating', 'episodes', 'members']]
)

In [7]:
from scipy.sparse import hstack

final_features = hstack([genre_matrix, numeric_features])

In [13]:
similarity_matrix = cosine_similarity(final_features)

In [9]:
# Create index mapping
anime_index = pd.Series(df.index, index=df['name']).drop_duplicates()

def recommend_anime(anime_name, top_n=5):
    if anime_name not in anime_index:
        return "Anime not found"
    
    idx = anime_index[anime_name]
    
    # Get similarity scores
    sim_scores = list(enumerate(similarity_matrix[idx]))
    
    # Sort by similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Remove itself
    sim_scores = sim_scores[1:top_n+1]
    
    anime_indices = [i[0] for i in sim_scores]
    
    return df[['name', 'genre', 'rating']].iloc[anime_indices]

In [10]:
print(recommend_anime("Naruto", top_n=5))

                                                   name  \
615                                  Naruto: Shippuuden   
1472        Naruto: Shippuuden Movie 4 - The Lost Tower   
1573  Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...   
486                            Boruto: Naruto the Movie   
1343                                        Naruto x UT   

                                                  genre  rating  
615   Action, Comedy, Martial Arts, Shounen, Super P...    7.94  
1472  Action, Comedy, Martial Arts, Shounen, Super P...    7.53  
1573  Action, Comedy, Martial Arts, Shounen, Super P...    7.50  
486   Action, Comedy, Martial Arts, Shounen, Super P...    8.03  
1343  Action, Comedy, Martial Arts, Shounen, Super P...    7.58  


In [11]:
def recommend_with_threshold(anime_name, threshold=0.5):
    if anime_name not in anime_index:
        return "Anime not found"
    
    idx = anime_index[anime_name]
    
    sim_scores = list(enumerate(similarity_matrix[idx]))
    
    # Filter by threshold
    sim_scores = [x for x in sim_scores if x[1] >= threshold and x[0] != idx]
    
    # Sort
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    anime_indices = [i[0] for i in sim_scores]
    
    return df[['name', 'genre', 'rating']].iloc[anime_indices]

In [12]:
print(recommend_with_threshold("Naruto", 0.3))
print(recommend_with_threshold("Naruto", 0.5))
print(recommend_with_threshold("Naruto", 0.7))

                                                    name  \
615                                   Naruto: Shippuuden   
1472         Naruto: Shippuuden Movie 4 - The Lost Tower   
1573   Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...   
486                             Boruto: Naruto the Movie   
1343                                         Naruto x UT   
...                                                  ...   
7623                           Shinsetsu Kachikachi Yama   
7642                                         Mugen Kouro   
10432                                      Super Backkom   
1355                       Hakkenden: Touhou Hakken Ibun   
1220                                         Accel World   

                                                   genre  rating  
615    Action, Comedy, Martial Arts, Shounen, Super P...    7.94  
1472   Action, Comedy, Martial Arts, Shounen, Super P...    7.53  
1573   Action, Comedy, Martial Arts, Shounen, Super P...    7.50  
486    Acti